# Week 2: One agent becomes a system

The Northwind support agent from Week 1, rebuilt on the Claude Agent SDK as a coordinator with four subagents.

Package: `claude-agent-sdk`. Model: whatever the SDK defaults to, overridable per agent.


## Setup


In [1]:
%pip install -q claude-agent-sdk anthropic python-dotenv

import asyncio
import json
import os
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

for candidate in [
    Path(".env"),
    Path("../week-1/.env"),
    Path("../week-1/demo-ui/.env.local"),
]:
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), "Put ANTHROPIC_API_KEY in .env"
print("ready")



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: /Users/akshika47/Documents/GitHub/AI-Internship/claude-architect-bootcamp/week-1/.venv/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


ready


Three small dicts stand in for the backend. Every demo tonight is about control flow, so the data stays deliberately simple.


In [2]:
TODAY = "2026-09-04"

CUSTOMERS = {
    "alice@example.com": {
        "customer_id": "CUST-001", "name": "Alice Chen", "tier": "Gold",
    },
}

ORDERS = {
    "ORD-123": {"customer_id": "CUST-001", "sku": "EL-4402", "amount": 149.99,
                "status": "delivered", "delivered_on": "2026-08-20",
                "condition": "damaged", "charges": 1},
    "ORD-456": {"customer_id": "CUST-001", "sku": "AP-1180", "amount": 89.00,
                "status": "delivered", "delivered_on": "2026-08-14",
                "condition": "ok", "charges": 2},
    "ORD-789": {"customer_id": "CUST-001", "sku": "FW-2201", "amount": 812.40,
                "status": "delivered", "delivered_on": "2026-07-26",
                "condition": "unworn", "charges": 1},
}

POLICY = {
    "2.1": "Standard return window: 30 calendar days from delivery date.",
    "2.2": "Footwear and apparel may be returned within 45 days if unworn "
           "with tags attached.",
    "2.3": "Electronics return window: 14 days.",
    "3.4": "Auto-escalate to Tier 2 if the customer mentions lawyer, BBB, "
           "chargeback, or social media.",
    "3.5": "Refund requests over $500 require Tier 2 approval.",
}

REFUND_LOG: list[dict] = []


In [3]:
ALICE_TICKET = (
    "I'm alice@example.com. "
    "ORD-123 arrived damaged and I want a refund. "
    "ORD-456 got charged to my card twice. "
    "I bought shoes on ORD-789 forty days ago. Can I still return them? "
    "If this isn't sorted today I'm filing a chargeback."
)
print(ALICE_TICKET)


I'm alice@example.com. ORD-123 arrived damaged and I want a refund. ORD-456 got charged to my card twice. I bought shoes on ORD-789 forty days ago. Can I still return them? If this isn't sorted today I'm filing a chargeback.


## Demo 0: The Week 1 agent meets the ticket

The Week 1 loop, unchanged, on the raw Messages API. The notebook is self-contained, and the failure appears before anything new does.


In [4]:
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-sonnet-4-6"

WEEK1_TOOLS = [
    {"name": "get_customer",
     "description": "Look up a customer by email. Returns customer_id, name, tier.",
     "input_schema": {"type": "object", "properties": {"email": {"type": "string"}},
                      "required": ["email"]}},
    {"name": "lookup_order",
     "description": "Look up an order by ID. Returns status, dates, amount, condition, charges.",
     "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                      "required": ["order_id"]}},
    {"name": "process_refund",
     "description": "Refund an amount against an order.",
     "input_schema": {"type": "object",
                      "properties": {"order_id": {"type": "string"},
                                     "amount": {"type": "number"}},
                      "required": ["order_id", "amount"]}},
    {"name": "escalate_to_human",
     "description": "Hand the case to a human agent with a structured summary.",
     "input_schema": {"type": "object",
                      "properties": {"customer_id": {"type": "string"},
                                     "root_cause": {"type": "string"},
                                     "recommended_action": {"type": "string"}},
                      "required": ["customer_id", "root_cause", "recommended_action"]}},
]


def week1_execute(name, args):
    if name == "get_customer":
        return CUSTOMERS.get(args["email"], {"error": "unknown customer"})
    if name == "lookup_order":
        return ORDERS.get(args["order_id"], {"error": "unknown order"})
    if name == "process_refund":
        REFUND_LOG.append(args)
        return {"ok": True, **args}
    if name == "escalate_to_human":
        return {"ok": True, "ticket": "ESC-9001", **args}


In [5]:
def week1_run_agent(user_message, max_turns=10):
    """Week 1's loop: branch on stop_reason, iterate every tool_use block."""
    messages = [{"role": "user", "content": user_message}]
    for _ in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=1024,
            system="You are a Northwind support agent. Verify the customer "
                   "first, then resolve their request.",
            tools=WEEK1_TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            text = "".join(b.text for b in resp.content if b.type == "text")
            return text, messages
        results = []
        for block in resp.content:
            if block.type == "tool_use":
                print(f"assistant: [tool_use] {block.name} {block.input}")
                out = week1_execute(block.name, block.input)
                results.append({"type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(out)})
        messages.append({"role": "user", "content": results})
    return "max turns reached", messages


In [6]:
REFUND_LOG.clear()
final, _ = week1_run_agent(ALICE_TICKET)
print("\nassistant:", final)


assistant: [tool_use] get_customer {'email': 'alice@example.com'}
assistant: [tool_use] lookup_order {'order_id': 'ORD-123'}
assistant: [tool_use] lookup_order {'order_id': 'ORD-456'}
assistant: [tool_use] lookup_order {'order_id': 'ORD-789'}


assistant: [tool_use] process_refund {'order_id': 'ORD-123', 'amount': 149.99}
assistant: [tool_use] process_refund {'order_id': 'ORD-456', 'amount': 89.0}


assistant: [tool_use] escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'ORD-789 (shoes, $812.40) was delivered 40 days ago, just outside the standard 30-day return window. Item is confirmed unworn. Customer is Gold tier and is threatening a chargeback if not resolved today.', 'recommended_action': 'Given Gold tier status, unworn condition, and urgency (chargeback threat), consider approving a return/refund as a goodwill exception. Prioritize same-day resolution.'}



assistant: Here's a full summary of everything done for you today, Alice:

---

### ✅ ORD-123 — Damaged Item
**Refunded: $149.99** — processed and on its way back to your original payment method (typically 3–5 business days).

### ✅ ORD-456 — Double Charge
**Refunded: $89.00** — the duplicate charge has been reversed.

### 🔄 ORD-789 — Late Return (Shoes, $812.40)
This is just outside our standard 30-day window, so I've escalated it to a senior agent under **ticket ESC-9001**, flagged as **priority/same-day**. I've noted your Gold tier status, the unworn condition, and the urgency. You should hear back very shortly.

---

Please don't file a chargeback — it can complicate and delay your refunds. We're actively working on all three issues and the ORD-789 escalation is flagged for same-day resolution. Is there anything else I can help with in the meantime?


The loop was correct. The answer was incomplete. Nothing in `stop_reason` can tell you the difference, and that is the problem the rest of tonight solves.


## Demo 1: The SDK owns the loop

The same four tools, plus one for reading policy, become an in-process MCP server. Then `query()` runs the loop.


In [7]:
from claude_agent_sdk import (
    query,
    tool,
    create_sdk_mcp_server,
    AgentDefinition,
    ClaudeAgentOptions,
    ClaudeSDKClient,
    HookMatcher,
    AssistantMessage,
    ResultMessage,
    ToolUseBlock,
)


def ok(payload: dict) -> dict:
    return {"content": [{"type": "text", "text": json.dumps(payload)}]}


def err(category: str, retryable: bool, message: str, **extra) -> dict:
    payload = {"ok": False, "errorCategory": category,
               "isRetryable": retryable, "message": message, **extra}
    return {"content": [{"type": "text", "text": json.dumps(payload)}],
            "is_error": True}


In [8]:
@tool("get_customer",
      "Look up one customer by email address. Returns customer_id, name and "
      "loyalty tier. Use this to verify identity before any order operation. "
      "Read only.",
      {"email": str})
async def get_customer(args: dict[str, Any]) -> dict[str, Any]:
    c = CUSTOMERS.get(args["email"])
    if c is None:
        return err("validation", False, "No customer with that email.")
    return ok(c)


@tool("lookup_order",
      "Look up one order by its ID. Returns status, delivered_on, amount, "
      "SKU, condition and how many times it was charged. Use for any "
      "question about the state of a specific order. Read only.",
      {"order_id": str})
async def lookup_order(args: dict[str, Any]) -> dict[str, Any]:
    o = ORDERS.get(args["order_id"])
    if o is None:
        return err("validation", False, "Unknown order id.")
    return ok({"order_id": args["order_id"], **o})


@tool("read_policy",
      "Read one clause of the Northwind support policy by section number, "
      "for example '2.2'. Use when a decision depends on written policy. "
      "Read only.",
      {"section": str})
async def read_policy(args: dict[str, Any]) -> dict[str, Any]:
    text = POLICY.get(args["section"])
    if text is None:
        return err("validation", False, f"No policy section {args['section']}.")
    return ok({"section": args["section"], "text": text})


@tool("process_refund",
      "Issue a refund of a given amount against one order. Mutating. Only "
      "call after the order has been looked up and the customer verified.",
      {"order_id": str, "amount": float})
async def process_refund(args: dict[str, Any]) -> dict[str, Any]:
    REFUND_LOG.append({"order_id": args["order_id"], "amount": args["amount"]})
    return ok({"ok": True, "refunded": args["amount"], "order_id": args["order_id"]})


@tool("escalate_to_human",
      "Hand the case to a human agent. Supply customer_id, root_cause and "
      "recommended_action, because the human cannot see this conversation.",
      {"customer_id": str, "root_cause": str, "recommended_action": str})
async def escalate_to_human(args: dict[str, Any]) -> dict[str, Any]:
    return ok({"ok": True, "ticket": "ESC-9001", **args})


In [9]:
northwind = create_sdk_mcp_server(
    name="northwind",
    version="1.0.0",
    tools=[get_customer, lookup_order, read_policy, process_refund, escalate_to_human],
)

# The server key is the middle segment of every tool name.
for t in ["get_customer", "lookup_order", "read_policy",
          "process_refund", "escalate_to_human"]:
    print(f"mcp__northwind__{t}")


mcp__northwind__get_customer
mcp__northwind__lookup_order
mcp__northwind__read_policy
mcp__northwind__process_refund
mcp__northwind__escalate_to_human


`setting_sources=[]` stops the SDK reading `CLAUDE.md`, skills and hooks from your machine. Every run tonight is then the same on every laptop. Week 3 turns this back on.


In [10]:
async def run(prompt: str, **overrides) -> str:
    """Run one query() and print spawns, subagent activity and the result."""
    opts = dict(
        mcp_servers={"northwind": northwind},
        # Session permission allowlist — pre-approves MCP for subagents too.
        allowed_tools=["Agent", "mcp__northwind__*"],
        setting_sources=[],
        # Without this, notebook runs hang forever waiting on a permission prompt.
        permission_mode="bypassPermissions",
        max_turns=12,
    )
    opts.update(overrides)
    # Keep coordinator tool *visibility* narrow when demos pass tools=COORDINATOR_TOOLS,
    # but never drop MCP from the session allowlist (that reintroduces hangs).
    if "agents" in opts:
        opts["allowed_tools"] = ["Agent", "mcp__northwind__*"]
    result_text = ""
    async for message in query(prompt=prompt, options=ClaudeAgentOptions(**opts)):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, ToolUseBlock):
                    # The current SDK names the spawn tool "Agent".
                    # Older versions named it "Task". The exam says "Task".
                    if block.name in ("Agent", "Task"):
                        print("spawned:", block.input.get("subagent_type"),
                              "|", str(block.input.get("prompt", ""))[:60])
                    else:
                        print("   tool:", block.name, block.input)
        if getattr(message, "parent_tool_use_id", None):
            print("      (inside a subagent)")
        if isinstance(message, ResultMessage):
            if message.subtype == "success":
                result_text = message.result or ""
            else:
                print(f"\n(ended: {message.subtype})")
                result_text = getattr(message, "result", None) or result_text
    print("\nRESULT\n", result_text)
    return result_text


In [11]:
_ = await run("Customer alice@example.com asks whether ORD-789 is still "
              "returnable. Today is " + TODAY + ".")


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: ToolSearch {'query': 'select:mcp__northwind__lookup_order,mcp__northwind__read_policy,mcp__northwind__get_customer', 'max_results': 5}


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}


   tool: mcp__northwind__read_policy {'section': '2.2'}


   tool: mcp__northwind__read_policy {'section': '2.1'}
   tool: mcp__northwind__read_policy {'section': '2.3'}



RESULT
 **Short answer: yes — most likely returnable, with a deadline of Tuesday, September 9, 2026 (5 days left).** But that rests on one assumption worth confirming before you tell Alice.

**The facts**
- Alice Chen, CUST-001, Gold tier — ORD-789 does belong to her (identity verified).
- Delivered 2026-07-26, so today is **day 40**. Status `delivered`, condition `unworn`, $812.40, charged once.

**Why the window matters**
The answer flips depending on which policy clause governs:

| Clause | Window | Deadline | Verdict at day 40 |
|---|---|---|---|
| 2.1 standard | 30 days | Aug 25 | **Expired** |
| 2.2 footwear/apparel | 45 days | **Sep 9** | **Still open** |
| 2.3 electronics | 14 days | Aug 9 | Expired |

So this is only returnable if the item is footwear or apparel. The SKU is `FW-2201`, and "FW" reads as footwear — combined with the `unworn` condition field, which only makes sense for something wearable, that's a strong signal. But it is an inference from a naming convention, n

## Demo 2: A subagent is a separate agent with a fresh context

Two specialists first. The other two arrive in Demo 3.


In [12]:
AGENTS = {
    "order-investigator": AgentDefinition(
        description="Looks up order status, dates, amounts, condition and "
                    "charge count. Use for any question about the state of "
                    "a specific order.",
        prompt="You investigate orders. Report status, delivered_on, amount, "
               "condition and charges for the order you are given. "
               "Call lookup_order once, then answer. Do not retry.",
        tools=["mcp__northwind__lookup_order"],
        maxTurns=4,
    ),
    "policy-analyst": AgentDefinition(
        description="Answers questions that depend on written Northwind "
                    "policy, such as return windows and escalation triggers.",
        prompt="You answer policy questions. Always cite the section number "
               "you read. Read at most two sections, then answer.",
        tools=["mcp__northwind__read_policy"],
        maxTurns=4,
    ),
}

# Visible to the coordinator only. Subagents still get tools from AgentDefinition.
# Session permission allowlist stays mcp__northwind__* inside run().
COORDINATOR_TOOLS = ["Agent", "mcp__northwind__get_customer"]


In [13]:
_ = await run(
    "Customer alice@example.com asks whether ORD-789 is still returnable. "
    "Today is " + TODAY + ". Use your subagents.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}


spawned: order-investigator | Look up order ORD-789 and report back every field you get: s
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | Read the Northwind support policy sections covering returns 
      (inside a subagent)


...

   tool: mcp__northwind__read_policy {'section': '3'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | Find the Northwind support policy section covering escalatio
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '4'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '1'}
      (inside a subagent)
      (inside a subagent)



RESULT
 **Yes — ORD-789 is still returnable, but the window closes in 5 days.**

Verified for Alice Chen (CUST-001, Gold):

| | |
|---|---|
| Delivered | 2026-07-26 |
| Days elapsed | **40** |
| SKU | FW-2201 (footwear) |
| Condition on file | unworn |
| Amount | $812.40 |

**The policy path:** §2.1 sets a standard 30-day window, which has already passed. But §2.2 grants footwear and apparel 45 days *"if unworn with tags attached."* At 40 days with condition recorded as unworn, ORD-789 qualifies under §2.2. **Deadline: 2026-09-09.**

**One caveat worth passing to Alice.** §2.2's extended window has two conjunctive conditions — unworn *and* tags attached. Our record confirms only "unworn"; the tags field isn't recorded either way. That's unverified, not negative, so I'd tell her the return is good to go and ask her to confirm the tags are still on before shipping, since that's what the 45-day window hangs on. If tags are off, she'd fall back to §2.1's 30 days, which expired on 2026-08-

A subagent receives its own prompt and the text the coordinator sends it. It does not receive the coordinator's conversation, the coordinator's system prompt, or anything another subagent found.


In [14]:
# WRONG: the prompt assumes the policy-analyst can see the order date.
_ = await run(
    "Use order-investigator to look up ORD-789. Then use policy-analyst to "
    "decide whether that order is still in its return window.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: order-investigator | Look up order ORD-789 using the lookup_order tool. Report ba
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | Determine whether order ORD-789 is still within its return w
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | I need to know whether any Northwind policy clause outside s
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.3'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '3.1'}
      (inside a subagent)
      (inside a subagent)



RESULT
 ## ORD-789

| Field | Value |
|---|---|
| Customer | CUST-001 |
| SKU | FW-2201 |
| Amount | 812.40 |
| Status | delivered |
| Delivered on | 2026-07-26 (40 days ago) |
| Condition | unworn |
| Times charged | 1 |

## Return window: most likely inside, at day 40 of 45

Two clauses govern:

- **2.1** — *"Standard return window: 30 calendar days from delivery date."*
- **2.2** — *"Footwear and apparel may be returned within 45 days if unworn with tags attached."*

Under 2.1 alone this order is **out** of window by 10 days. It only survives via the 45-day footwear exception in 2.2 — which it appears to meet at day 40, with condition confirmed as "unworn."

## Two things I could not verify

**Tags.** Clause 2.2 requires unworn **and** tags attached. The order system confirms only "unworn"; it says nothing about tags. If tags are missing, 2.2 doesn't apply, the order falls back to the 30-day standard, and it is **outside** its window. This needs to be asked before a return is appro

In [15]:
# RIGHT: every fact the policy-analyst needs is stated in the prompt.
_ = await run(
    "Use order-investigator to look up ORD-789. Then invoke policy-analyst "
    "with a prompt that states the SKU, the delivered_on date, and that "
    "today is " + TODAY + ", and asks whether the return window has passed.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: order-investigator | Look up order ORD-789 and report all returned fields verbati
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | Question about Northwind return policy for a specific order.
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)
      (inside a subagent)



RESULT
 Both steps are done. Here's what came back.

**ORD-789 (order-investigator)**
| Field | Value |
|---|---|
| customer_id | CUST-001 |
| sku | FW-2201 |
| amount | 812.40 |
| status | delivered |
| delivered_on | 2026-07-26 |
| condition | unworn |
| charges | 1 (no double-charge) |

**Return window (policy-analyst): still open, but conditionally.**

40 calendar days have elapsed since delivery.

- **§2.1** — standard window is 30 days from delivery, which would have closed 2026-08-25, ten days ago.
- **§2.2** — footwear and apparel get 45 days if unworn *with tags attached*, moving the deadline to **2026-09-09**, five days out.

So the answer hinges entirely on §2.2 applying, and the analyst flagged two gaps you should close before treating the window as open:

1. **Tag status is unconfirmed.** The order record says "unworn" but says nothing about tags. §2.2 requires both. No tags means it falls back to §2.1 and the window closed ten days ago.
2. **The footwear category was inf

Parallel spawning is several `Agent` calls emitted in one coordinator response. Time it both ways.


In [16]:
t0 = time.time()
_ = await run(
    "Look up ORD-123, ORD-456 and ORD-789. Spawn one order-investigator per "
    "order in a single response so all three run at the same time. Report "
    "the three amounts.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)
parallel = time.time() - t0

t0 = time.time()
_ = await run(
    "Look up ORD-123, then when that is done look up ORD-456, then when that "
    "is done look up ORD-789. Use one order-investigator at a time. Report "
    "the three amounts.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)
sequential = time.time() - t0

print(f"\nparallel   {parallel:.1f}s\nsequential {sequential:.1f}s")


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: order-investigator | Look up order ORD-123 and report its full details: status, d
      (inside a subagent)


spawned: order-investigator | Look up order ORD-456 and report its full details: status, d
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-123'}
      (inside a subagent)
      (inside a subagent)


spawned: order-investigator | Look up order ORD-789 and report its full details: status, d
      (inside a subagent)


...

spawned: order-investigator | Look up order ORD-456 and report its full details: status, d
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}
      (inside a subagent)
      (inside a subagent)


spawned: order-investigator | Look up order ORD-789 and report its full details: status, d
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}
      (inside a subagent)
      (inside a subagent)



RESULT
 All three lookups are complete, run one at a time in order.

**Amounts:**
| Order | Amount |
|---|---|
| ORD-123 | $149.99 |
| ORD-456 | $89.00 |
| ORD-789 | $812.40 |

Total: **$1,051.39**

Two things surfaced along the way that you may want to act on, though I've made no changes:
- **ORD-123** — delivered 2026-08-20 in **damaged** condition (charged once).
- **ORD-456** — delivered fine, but **charged twice** for a single $89.00 order, suggesting a possible $89.00 double-billing.

ORD-789 looks clean (delivered 2026-07-26, unworn, charged once), though it's ~40 days out if any return window applies. Let me know if you'd like me to check the relevant policy sections or process anything.

parallel   17.0s
sequential 37.4s


## Demo 3: One job and one contract per subagent


In [17]:
_ = await run(
    "Ask policy-analyst whether footwear delivered on 2026-07-26 can still "
    "be returned today, " + TODAY + ". Return exactly what it says.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: policy-analyst | Question: Can footwear that was delivered on 2026-07-26 stil
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)



RESULT
 Here is exactly what the policy-analyst said:

---

**Yes** — the return is still within the window.

**Policy basis:**
- **Section 2.1** — Standard return window: 30 calendar days from delivery date.
- **Section 2.2** — Footwear and apparel may be returned within **45 days** if unworn with tags attached.

Footwear falls under the category-specific rule in 2.2, so the applicable window is **45 days**, not the standard 30.

**Day count:** Delivery 2026-07-26 → today 2026-09-04 = **40 days** (5 days remaining in July + 31 in August + 4 in September).

**Result:** 40 of 45 days used. The return is accepted, with the deadline falling on **2026-09-09** — 5 days from today.

One condition attaches: Section 2.2 grants the extended 45-day window only for items that are **unworn with tags attached**. If the footwear has been worn or the tags removed, the 2.2 extension does not apply and the item falls back to the Section 2.1 standard window of 30 days, which expired on 2026-08-25 — in 

In [18]:
FINDING_CONTRACT = """

Return a JSON array and nothing else. Each element must contain:
  "concern_id"  the concern this finding answers, or "none" if not given one
  "claim"       one sentence stating what you found
  "evidence"    the exact field value or policy clause you read
  "source"      the tool call or policy section it came from
"""

AGENTS["order-investigator"] = AgentDefinition(
    description=AGENTS["order-investigator"].description,
    prompt="You investigate orders." + FINDING_CONTRACT,
    tools=["mcp__northwind__lookup_order"],
    maxTurns=4,
)
AGENTS["policy-analyst"] = AgentDefinition(
    description=AGENTS["policy-analyst"].description,
    prompt="You answer policy questions." + FINDING_CONTRACT,
    tools=["mcp__northwind__read_policy"],
    maxTurns=4,
)


In [19]:
_ = await run(
    "Concern c3: footwear on ORD-789, delivered 2026-07-26, today is "
    + TODAY + ". Ask policy-analyst whether it is still returnable and "
    "return its JSON unchanged.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: policy-analyst | Question about Northwind written policy for concern c3.

Fac
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.3'}
      (inside a subagent)
      (inside a subagent)



RESULT
 Here is the policy-analyst's JSON, unchanged:

```json
{
  "concern_id": "c3",
  "order_id": "ORD-789",
  "category": "footwear",
  "delivered_on": "2026-07-26",
  "today": "2026-09-04",
  "days_since_delivery": 40,
  "policy_sections_cited": ["2.1", "2.2", "2.3"],
  "return_window_days": 45,
  "returnable": true,
  "reasoning": "Section 2.1 sets a standard 30-day return window, but section 2.2 provides a category-specific override: 'Footwear and apparel may be returned within 45 days if unworn with tags attached.' ORD-789 is footwear, so the 45-day window applies. Delivery was 2026-07-26 and today is 2026-09-04, which is 40 days later, within the 45-day limit. Section 2.3 (14-day electronics window) does not apply. Returnability is conditional: section 2.2 requires the item be unworn with tags attached, and that condition was not provided in the facts and must be verified before the return is approved."
}
```

One flag worth noting: the analyst's `returnable: true` is conditi

Now the other half of a subagent's job description: what it is allowed to touch. The `tools` field is left out on purpose here.


In [20]:
LOOSE_AGENTS = {
    "order-investigator": AgentDefinition(
        description="Looks up order status, dates and amounts.",
        prompt="You investigate orders. You never issue refunds.",
        maxTurns=4,
        # tools omitted, so this agent inherits every tool available
    ),
}

REFUND_LOG.clear()
_ = await run(
    "Use order-investigator to check ORD-456, which the customer says was "
    "charged twice, and put the customer right.",
    agents=LOOSE_AGENTS,
    allowed_tools=["Agent", "mcp__northwind__*"],
)
print("\nREFUND_LOG:", REFUND_LOG)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: order-investigator | Investigate order ORD-456. The customer reports they were ch
      (inside a subagent)


   tool: ToolSearch {'query': 'select:mcp__northwind__lookup_order,mcp__northwind__get_customer,mcp__northwind__read_policy', 'max_results': 5}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)


...

      (inside a subagent)
   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
      (inside a subagent)
      (inside a subagent)


      (inside a subagent)


   tool: ToolSearch {'query': 'select:mcp__northwind__escalate_to_human', 'max_results': 1}


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Alice Chen (CUST-001, Gold) reports ORD-456 ($89.00, SKU AP-1180, delivered 2026-08-14) was charged to her card twice. System state corroborates: the order record shows charges=2 against a single $89.00 one-item order, with no offsetting credit. Order is delivered/condition ok, so there is no partial shipment, replacement or reshipment that would legitimately explain a second charge.\n\nIMPORTANT EVIDENCE GAP: the order-lookup tool exposes only a scalar charges count. There are no itemized transaction records available to me — no transaction IDs, timestamps, per-charge amounts, or authorization-vs-settlement flags. I therefore CANNOT distinguish a genuine double settlement from one settled charge plus a pending authorization hold. Those look identical on a cardholder statement but call for opposite responses (refund vs. no action, hold drops off in 3-5 business days). Resolving this requires payment-p


RESULT
 Investigated and escalated — **ticket ESC-9001**. I did not issue a refund. Here's why, and what you need to decide.

**What ORD-456 shows**

| | |
|---|---|
| Customer | Alice Chen — CUST-001, Gold tier, alice@example.com |
| Item | AP-1180 (apparel), $89.00 |
| Status | delivered 2026-08-14, condition ok |
| Charges | **2** |

Identity is confirmed — `get_customer` on her email returns CUST-001, matching the order.

**The claim is corroborated, but not proven**

The record shows `charges: 2` on a single $89 order with no offsetting credit, and the order is cleanly delivered — no partial shipment or replacement that would justify a second charge. So her complaint matches system state independently.

But the order tool exposes only a bare integer count. There are no transaction records behind it — no IDs, timestamps, or authorization-vs-settlement flags. That means **I cannot tell a genuine double settlement from one charge plus a pending authorization hold.** Those are indist

In [21]:
REFUND_LOG.clear()

AGENTS["refund-processor"] = AgentDefinition(
    description="Issues refunds against orders that have already been "
                "investigated and the customer verified. Use only to "
                "execute a refund.",
    prompt="You issue the refund you are asked for." + FINDING_CONTRACT,
    tools=["mcp__northwind__process_refund"],
    maxTurns=4,
)

AGENTS["escalation-writer"] = AgentDefinition(
    description="Compiles a structured handoff for a human agent. Use when "
                "policy requires a human, or the automated system cannot act.",
    prompt="You write escalation handoffs. Include customer_id, root_cause "
           "and recommended_action, because the human cannot see this "
           "conversation." + FINDING_CONTRACT,
    tools=["mcp__northwind__escalate_to_human"],
    maxTurns=4,
)

for name, agent in AGENTS.items():
    print(f"{name:>20}  {agent.tools}")


  order-investigator  ['mcp__northwind__lookup_order']
      policy-analyst  ['mcp__northwind__read_policy']
    refund-processor  ['mcp__northwind__process_refund']
   escalation-writer  ['mcp__northwind__escalate_to_human']


In [22]:
# Same request as cell 32, now against the scoped investigator.
_ = await run(
    "Use order-investigator to check ORD-456, which the customer says was "
    "charged twice, and put the customer right.",
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)
print("\nREFUND_LOG:", REFUND_LOG)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


spawned: order-investigator | Look up order ORD-456 and report back everything: status, de
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}
      (inside a subagent)
      (inside a subagent)


spawned: policy-analyst | Find and report the Northwind support policy governing dupli
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '1.1'}
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
      (inside a subagent)
      (inside a subagent)


...

   tool: mcp__northwind__read_policy {'section': '2.4'}
   tool: mcp__northwind__read_policy {'section': '2.5'}


   tool: mcp__northwind__read_policy {'section': '4.1'}


   tool: mcp__northwind__read_policy {'section': '4.2'}
   tool: mcp__northwind__read_policy {'section': '3.2'}
   tool: mcp__northwind__read_policy {'section': '1.2'}


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Order ORD-456 (SKU AP-1180, amount 89.00, status delivered, delivered_on 2026-08-14, condition "ok") shows "charges": 2 in the order record. The customer was billed twice for a single order. This is a confirmed billing/payment-capture error, not a return or product-condition issue.', 'recommended_action': 'Refund 89.00 against ORD-456 to reverse the duplicate charge, leaving one valid charge of 89.00. Two reasons this was not auto-processed: (1) No customer email was supplied, so identity could not be verified via get_customer — CUST-001 was read off the order record, which is not independent verification. Please verify the requester owns CUST-001 before releasing funds. (2) The Northwind policy set contains only sections 2.1, 2.2 and 2.3, all return-window rules (30-day standard, 45-day apparel, 14-day electronics). None govern duplicate charges, so no written clause authorises an automated refund he


RESULT
 **The customer is right — ORD-456 was charged twice.** I've raised ticket **ESC-9001** recommending a $89.00 refund.

What the order shows:

| Field | Value |
|---|---|
| Status | delivered (2026-08-14) |
| Amount | 89.00 |
| SKU | AP-1180 |
| Condition | ok |
| **Charges** | **2** |

I stopped short of issuing the refund myself, for two reasons worth your attention:

**I couldn't verify identity.** `process_refund` requires a verified customer, and `get_customer` looks up by email — which I wasn't given. The `CUST-001` ID came off the order record itself, so using it to "verify" the person asking would be circular. **If you give me the customer's email, I can verify and refund immediately**, and you can close ESC-9001.

**Policy is silent here.** I read every section that exists — 2.1, 2.2, 2.3 — and they're all return windows (30-day standard, 45-day apparel, 14-day electronics). Nothing covers duplicate charges. I didn't want to stretch a return clause into refund authority

## Demo 4: A hook is a rule that runs on every tool call

Scoping decided which agent holds `process_refund`. The $500 ceiling still has to hold, and it has to hold once, not once per agent.


In [60]:
REFUND_CEILING = 500.00


async def enforce_ceiling(input_data, tool_use_id, context):
    """PreToolUse hook. Runs before process_refund executes, in any agent."""
    amount = input_data["tool_input"].get("amount", 0)
    if amount <= REFUND_CEILING:
        return {}
    return {
        "hookSpecificOutput": {
            "hookEventName": input_data["hook_event_name"],
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                f"${amount:.2f} is over the ${REFUND_CEILING:.2f} automatic "
                "ceiling, policy 3.5. Escalate to a human agent instead."
            ),
        }
    }


HOOKS = {
    "PreToolUse": [
        HookMatcher(matcher="mcp__northwind__process_refund",
                    hooks=[enforce_ceiling]),
    ]
}


In [61]:
async def run_client(prompt: str) -> str:
    """Same as run(), using ClaudeSDKClient, which is the documented hook path."""
    options = ClaudeAgentOptions(
        mcp_servers={"northwind": northwind},
        agents=AGENTS,
        tools=COORDINATOR_TOOLS,
        allowed_tools=["Agent", "mcp__northwind__*"],
        hooks=HOOKS,
        setting_sources=[],
        permission_mode="bypassPermissions",
        max_turns=12,
    )
    result_text = ""
    async with ClaudeSDKClient(options=options) as client:
        await client.query(prompt)
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        label = "spawned:" if block.name in ("Agent", "Task") else "   tool:"
                        print(label, block.name, block.input)
            if isinstance(message, ResultMessage):
                if message.subtype == "success":
                    result_text = message.result or ""
                else:
                    print(f"\n(ended: {message.subtype})")
                    result_text = getattr(message, "result", None) or result_text
    print("\nRESULT\n", result_text)
    return result_text


In [ ]:
REFUND_LOG.clear()
_ = await run_client(
    "Customer alice@example.com wants a full refund of 812.40 on ORD-789. "
    "Verify them, investigate the order, then refund it."
)
print("\nREFUND_LOG:", REFUND_LOG)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}
spawned: Agent {'description': 'Check refund policy', 'subagent_type': 'policy-analyst', 'prompt': 'I need the Northwind policy rules governing a refund decision on this case:\n\n- Customer: CUST-001, Alice Chen, loyalty tier Gold\n- Order ORD-789, SKU FW-2201, amount 812.40\n- Status: delivered on 2026-07-26 (today is 2026-09-05, so 41 days ago)\n- Condition: unworn\n- Times charged: 1\n\nPlease read the relevant policy sections and tell me:\n1. What is the standard return/refund window after delivery, and does this order fall inside or outside it?\n2. Are there any exceptions or extended windows for Gold tier customers?\n3. Are there any thresholds (e.g. refund amount) or other conditions that require human escalation or approval rather than an automated refund?\n4. Your bottom line: can an agent process this 812.40 refund automatically, or must it be esc

In [26]:
REFUND_LOG.clear()
_ = await run_client(
    "Customer alice@example.com wants a refund of 149.99 on ORD-123, which "
    "arrived damaged. Verify them, investigate the order, then refund it."
)
print("\nREFUND_LOG:", REFUND_LOG)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-123'}


spawned: Agent {'description': 'Check refund policy for damaged item', 'subagent_type': 'policy-analyst', 'prompt': 'I need to determine whether an automated refund is permitted for this case, per written Northwind policy.\n\nCase facts:\n- Customer: CUST-001, Alice Chen, Gold tier\n- Order ORD-123, SKU EL-4402, amount 149.99\n- Status: delivered on 2026-08-20 (today is 2026-09-04, so 15 days ago)\n- Condition: damaged\n- Charges: 1 (single charge, no duplicate billing)\n- Customer requests a full refund of 149.99\n\nPlease read the relevant policy sections and report:\n1. The return/refund window for damaged goods, and whether 15 days post-delivery is inside it.\n2. Any refund amount threshold above which a refund requires human approval or escalation.\n3. Any escalation triggers that apply to these facts.\n4. Any tier-specific (Gold) provisions.\n5. A clear bottom line: may an automated agent refund 149.99 on this order, or must it be escalated?\n\nQuote the section numbers and claus

   tool: mcp__northwind__read_policy {'section': '1'}
   tool: mcp__northwind__read_policy {'section': '2'}


   tool: mcp__northwind__read_policy {'section': '2.1'}
   tool: mcp__northwind__read_policy {'section': '2.2'}


...

   tool: mcp__northwind__read_policy {'section': '3'}


   tool: mcp__northwind__read_policy {'section': '4'}


   tool: mcp__northwind__read_policy {'section': '1'}
   tool: mcp__northwind__read_policy {'section': '2'}


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Alice Chen (CUST-001, Gold tier) requests a full refund of 149.99 on ORD-123, SKU EL-4402, which arrived damaged. Identity verified by email alice@example.com; order confirmed to belong to this customer. Order lookup confirms: status delivered, delivered_on 2026-08-20, amount 149.99, condition "damaged", charges 1 (no duplicate billing). The refund is NOT auto-approvable: policy 2.3 sets a 14-day return window for electronics, and delivery was 15 days before today (2026-09-04) — one day outside the window. I reviewed the full readable policy (only sections 2.1, 2.2 and 2.3 exist); there is no damaged-on-arrival clause that overrides or extends the window, and no Gold-tier exception. Automated refund therefore withheld pending human judgment.', 'recommended_action': 'Recommend approving the 149.99 refund as a goodwill exception. Mitigating factors: the item is confirmed damaged in the order record (not


RESULT
 **I did not process the refund.** It's escalated to a human as ticket **ESC-9001**.

Here's what I found:

**Verification — passed.** alice@example.com resolves to Alice Chen, CUST-001, Gold tier. ORD-123 belongs to that same customer ID, so the request is legitimate.

**Order — claim confirmed.** ORD-123, SKU EL-4402, 149.99, status delivered, condition `damaged`, charged once. Her description of the problem is accurate and there's no double-billing wrinkle.

**Policy — this is where it stopped.** Section 2.3 sets a **14-day** return window for electronics. Delivery was 2026-08-20; today is 2026-09-04. That's **15 days — one day late.**

I looked hard for a way to say yes. The entire policy is three clauses (2.1 standard 30 days, 2.2 footwear/apparel 45 days, 2.3 electronics 14 days). There is **no damaged-on-arrival exception and no Gold-tier exception** — I probed for those sections specifically and they don't exist. The shorter 14-day electronics window is the one that gov

`PreToolUse` sees the request and can deny it or rewrite its input. `PostToolUse` sees the result and can rewrite what the model reads. Normalising timestamps and status codes from different tools is a `PostToolUse` job.


## Demo 5: When a subagent fails, send the failure up with context

The order service goes down for ORD-456. Two versions of the failure, and the coordinator's ability to recover depends entirely on which one it gets.


In [ ]:
FAIL_ORDER = "ORD-456"


@tool("lookup_order",
      "Look up one order by its ID. Returns status, delivered_on, amount, "
      "SKU, condition and charge count. Read only.",
      {"order_id": str})
async def lookup_order_generic(args: dict[str, Any]) -> dict[str, Any]:
    if args["order_id"] == FAIL_ORDER:
        # WRONG: two words. The coordinator cannot choose between
        # retry, skip and escalate from this.
        return {"content": [{"type": "text", "text": "Lookup failed."}],
                "is_error": True}
    o = ORDERS.get(args["order_id"])
    return ok({"order_id": args["order_id"], **o})


northwind_generic = create_sdk_mcp_server(
    name="northwind", version="1.0.0",
    tools=[get_customer, lookup_order_generic, read_policy,
           process_refund, escalate_to_human],
)

_ = await run(
    "Customer alice@example.com says ORD-456 was charged twice. Investigate "
    "and resolve it.",
    mcp_servers={"northwind": northwind_generic},
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}


spawned: policy-analyst | I'm handling a customer claim that order ORD-456 was charged
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.1'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)


...

   tool: mcp__northwind__read_policy {'section': '5.1'}
   tool: mcp__northwind__read_policy {'section': '3.2'}


   tool: mcp__northwind__read_policy {'section': '2.5'}
   tool: mcp__northwind__read_policy {'section': '2.6'}


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Customer Alice Chen (CUST-001, Gold tier, alice@example.com) reports that order ORD-456 was charged twice. The claim could NOT be verified: lookup_order for ORD-456 returned "Lookup failed" on three separate attempts, so the order\'s charge count, amount, status and SKU are all unknown to the automated system. Identity verification succeeded via get_customer. A policy review found only sections 2.1 (30-day standard return window), 2.2 (footwear/apparel 45 days) and 2.3 (electronics 14 days) to be readable; sections 1.x, 3.x, 4.x and 5.x do not exist. There is no accessible written policy governing duplicate-charge claims or authorizing a refund when order details cannot be verified. No refund was issued, because the duplicate charge is unconfirmed and the correct refund amount is unknown.', 'recommended_action': "1) Pull ORD-456 directly from the order/billing system out-of-band to confirm whether two


RESULT
 I escalated this to a human rather than resolving it automatically. Ticket **ESC-9001**.

**What I found:**

- **Identity verified** — Alice Chen, CUST-001, Gold tier.
- **Order ORD-456 could not be retrieved.** `lookup_order` returned "Lookup failed" on three separate attempts. That means the charge count, amount, status and SKU are all unknown to me — I have no way to confirm the double charge, and no way to know what a correct refund amount would even be.
- **No applicable policy exists.** Only sections 2.1, 2.2 and 2.3 are readable, and all three are return-window rules (30 days standard, 45 for apparel, 14 for electronics). Sections 1.x, 3.x, 4.x and 5.x don't exist. Nothing covers duplicate charges or authorizes acting on an unverified claim.

**Why I didn't refund:** issuing a refund here would mean guessing both that a second charge occurred and how much it was, based only on the customer's report. That's a mutating action against an order I could not read. A wrong amo

In [ ]:
@tool("lookup_order",
      "Look up one order by its ID. Returns status, delivered_on, amount, "
      "SKU, condition and charge count. Read only.",
      {"order_id": str})
async def lookup_order_structured(args: dict[str, Any]) -> dict[str, Any]:
    if args["order_id"] == FAIL_ORDER:
        # RIGHT: category, retryability, what was attempted, what survived,
        # and what the coordinator could do next.
        return err(
            "transient", True, "Order service timed out.",
            attempted=f"lookup_order({args['order_id']})",
            partialResults=[{"concern_id": "c2",
                             "claim": "ORD-456 exists on the account."}],
            alternatives=["retry once",
                          "proceed with the other concerns and flag c2"],
        )
    o = ORDERS.get(args["order_id"])
    return ok({"order_id": args["order_id"], **o})


northwind_structured = create_sdk_mcp_server(
    name="northwind", version="1.0.0",
    tools=[get_customer, lookup_order_structured, read_policy,
           process_refund, escalate_to_human],
)

_ = await run(
    "Customer alice@example.com says ORD-456 was charged twice. Investigate "
    "and resolve it. If a lookup fails, say what was attempted, what you "
    "know anyway, and what you did about it.",
    mcp_servers={"northwind": northwind_structured},
    agents=AGENTS, tools=COORDINATOR_TOOLS,
)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}
   tool: mcp__northwind__read_policy {'section': '2.2'}


spawned: policy-analyst | Find the Northwind support policy clause(s) covering duplica
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '1.1'}
      (inside a subagent)
      (inside a subagent)


...

   tool: mcp__northwind__read_policy {'section': '2.2'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '3.1'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.3'}
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.4'}
      (inside a subagent)
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '2.5'}
      (inside a subagent)
      (inside a subagent)
   tool: mcp__northwind__read_policy {'section': '2.6'}
      (inside a subagent)
      (inside a subagent)


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Customer Alice Chen (CUST-001, Gold tier, alice@example.com) reports ORD-456 was charged twice. The claim could not be verified: lookup_order(ORD-456) timed out twice (transient error, retried once as the error advised). Both failures returned a partial result indicating ORD-456 does exist on the account, but no status, amount, SKU, condition, or charge_count was retrievable. Without charge_count the duplicate charge is unconfirmed, and without the order amount no refund figure can be determined, so no refund was issued. The written policy offers no guidance here: the full policy corpus is sections 2.1-2.3, which cover only return windows (30-day standard, 45-day footwear/apparel, 14-day electronics). There is no clause on duplicate or double charges and no defined escalation trigger; sections 1, 3, and 2.4+ do not exist.', 'recommended_action': '1) Re-run the order lookup for ORD-456 once the order s


RESULT
 ## Result: escalated to a human — ticket **ESC-9001**. No refund issued.

**What I attempted**
- ✅ Verified identity: alice@example.com → Alice Chen, CUST-001, Gold tier.
- ❌ `lookup_order(ORD-456)` — timed out. Error was marked transient/retryable, so I retried once. Timed out again.
- ✅ Read policy — the entire corpus is sections 2.1–2.3 (return windows only). Sections 1.x, 3.x and 2.4+ don't exist.

**What I know anyway**
Both failed lookups returned a partial result confirming **ORD-456 does exist on Alice's account**. That is all. Status, amount, SKU, condition and — critically — `charge_count` were all unavailable.

**Why I stopped short of a refund**
Two things were missing, each disqualifying on its own:
1. **The duplicate is unconfirmed.** `charge_count` is the field that proves a double charge, and I never retrieved it. The customer's report is plausible but not evidence.
2. **There is no amount to refund.** A refund requires a figure, and inventing one against an un

An access failure and an empty result are different answers. A query that could not run may be worth retrying. A query that ran and matched nothing returns the same nothing every time.

In Python the flag is `is_error`. In TypeScript and the MCP specification it is `isError`, which is the spelling on the exam.


## Demo 6: The coordinator decides what gets covered

Back to the full ticket, with all four specialists available and the healthy server.


In [29]:
NARROW_SYSTEM = (
    "You are a Northwind support coordinator. Verify the customer, look up "
    "any orders they mention, and process refunds where they are due."
)

REFUND_LOG.clear()
_ = await run(ALICE_TICKET, agents=AGENTS, tools=COORDINATOR_TOOLS,
              system_prompt=NARROW_SYSTEM)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-123'}


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-456'}
   tool: mcp__northwind__lookup_order {'order_id': 'ORD-789'}


spawned: policy-analyst | Read the Northwind support policy and report the exact rules
      (inside a subagent)


   tool: mcp__northwind__read_policy {'section': '1'}
      (inside a subagent)
      (inside a subagent)


...

   tool: mcp__northwind__read_policy {'section': '3.9'}
   tool: mcp__northwind__read_policy {'section': '3.10'}


   tool: mcp__northwind__read_policy {'section': '3.0'}
   tool: mcp__northwind__read_policy {'section': '2.8'}


   tool: mcp__northwind__process_refund {'order_id': 'ORD-456', 'amount': 89}


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Three-order case for Alice Chen (Gold), verified via alice@example.com; all three orders confirmed under CUST-001. Two mandatory Tier 2 triggers fired. (1) Policy 3.4 - customer explicitly threatened a chargeback ("If this isn\'t sorted today I\'m filing a chargeback"), which is an unconditional auto-escalate trigger. (2) Policy 3.5 - ORD-789 refund value is $812.40, over the $500 Tier 2 approval threshold. Order detail: ORD-123, SKU EL-4402, $149.99, delivered 2026-08-20, condition DAMAGED, 1 charge - electronics window under 2.3 is 14 days and today (2026-09-04) is day 15, so it is 1 day outside the return window, BUT the item arrived damaged and no damaged-on-arrival / defect clause exists in the retrievable policy (probed 1.x, 2.4-2.8, 3.0-3.3, 3.6-3.10, 4.x, 5.x, 6.x, 7.x - only 2.1, 2.2, 2.3, 3.4, 3.5 exist). A damage claim is arguably not a discretionary "return" at all, so the 14-day window ma


RESULT
 Hi Alice — verified you as Alice Chen (CUST-001, Gold), and all three orders are confirmed on your account. Here's where each one landed.

**ORD-456 — refunded, done.** Your card really was charged twice ($89.00 × 2). A duplicate charge is a billing error, not a return, so no return window applies to it. **I've refunded $89.00.**

**ORD-789 (shoes) — yes, you can still return them.** This was your actual question, and the answer is good news. Footwear gets a 45-day window if unworn with tags attached (policy 2.2), not the standard 30. You're on **day 40, so you have until 2026-09-09** — please get them moving before then. One catch: at $812.40 the refund is over the $500 mark where a Tier 2 approver has to sign off (policy 3.5), so I can't release that one myself. It's an approval step, not an eligibility problem. Make sure the tags are still attached, since that's part of the rule.

**ORD-123 (damaged) — I've pushed for it, but a human has to rule.** I want to be straight wit

The specialists did exactly what they were given. The coordinator gave them two concerns out of four. The system prompt described a procedure, and the procedure only mentioned refunds.


In [30]:
GOAL_SYSTEM = (
    "You are a Northwind support coordinator. Resolve every concern in the "
    "customer's message. First list the distinct concerns. Each concern must "
    "be handled by a subagent or explicitly escalated. Do not finish with "
    "any concern unassigned. Verify the customer before any order operation."
)

REFUND_LOG.clear()
_ = await run(ALICE_TICKET, agents=AGENTS, tools=COORDINATOR_TOOLS,
              system_prompt=GOAL_SYSTEM)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}


spawned: order-investigator | Look up order ORD-123 and report every field returned: statu
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-123'}
      (inside a subagent)


      (inside a subagent)
spawned: order-investigator | Look up order ORD-456 and report every field returned: statu
      (inside a subagent)


...

spawned: refund-processor | Process a refund of 89.00 against order ORD-456.

Authorizat
      (inside a subagent)


   tool: mcp__northwind__process_refund {'order_id': 'ORD-456', 'amount': 89}
      (inside a subagent)
      (inside a subagent)


spawned: escalation-writer | File an escalation to a human agent using customer_id "CUST-
      (inside a subagent)


   tool: mcp__northwind__escalate_to_human {'customer_id': 'CUST-001', 'root_cause': 'Alice Chen (CUST-001, Gold tier, alice@example.com) reports order ORD-123 arrived damaged and requests a refund of the full 149.99. The order record corroborates the claim: status delivered, delivered_on 2026-08-20, amount 149.99, SKU EL-4402 (electronics), condition "damaged", charges 1. The blocker is that the Northwind policy has no clause governing refunds for items that arrive damaged — the policy contains only return-window clauses (2.1 standard 30 days, 2.2 footwear 45 days unworn, 2.3 electronics 14 days). The only adjacent clause, 2.3, gives electronics a 14-day return window, and delivery was 15 days before today (2026-09-04), placing this one day outside that window. A damaged-on-arrival defect claim is arguably not an ordinary return and the system-recorded condition of "damaged" independently supports the customer, but no written authority exists for an automated agent to approve this 149


RESULT
 Hi Alice — verified you as Alice Chen (CUST-001, Gold tier), and all three orders are on your account. Here's where each of your four concerns landed.

**ORD-456 — double charge: resolved, money already back.**
Confirmed: the order is $89.00 and the charge counter reads 2, so you were billed $178.00. I've refunded $89.00, reversing exactly the duplicate charge (confirmation: `refunded: 89`). This one was clear-cut and needed no approval.

**ORD-789 — shoes, 40 days: yes, you can still return them.**
Delivered 2026-07-26, so today is day 40. Policy clause 2.2 says *"Footwear and apparel may be returned within 45 days if unworn with tags attached."* Your item is SKU FW-2201 (footwear) and its condition is recorded as **unworn**. You're inside the window with **5 days left — through 2026-09-09**. One caveat: 2.2 also requires tags attached, which isn't in our records, so keep the tags on. Tell me if you'd like me to start the return.

**ORD-123 — damaged item: escalated to a huma

A better prompt raised the odds. It did not make coverage deterministic. The check below does, by comparing the coordinator's own decomposition against the findings that came back.


In [31]:
async def extract_concerns(ticket: str) -> list[dict]:
    """The coordinator's decomposition, returned as data instead of prose."""
    raw = await run(
        "List every distinct concern in this customer message as a JSON "
        "array and nothing else. Each element: {\"id\": \"c1\", "
        "\"kind\": \"...\", \"detail\": \"...\"}. Message:\n" + ticket,
        allowed_tools=[],
    )
    start, end = raw.find("["), raw.rfind("]") + 1
    return json.loads(raw[start:end])


concerns = await extract_concerns(ALICE_TICKET)
for c in concerns:
    print(c)


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors



RESULT
 [
  {"id": "c1", "kind": "damaged_item_refund", "detail": "ORD-123 arrived damaged; customer requests a refund."},
  {"id": "c2", "kind": "duplicate_charge", "detail": "ORD-456 was charged to the customer's card twice."},
  {"id": "c3", "kind": "return_eligibility_question", "detail": "Shoes bought on ORD-789 forty days ago; customer asks whether they can still be returned."},
  {"id": "c4", "kind": "chargeback_threat", "detail": "Customer will file a chargeback if the issues are not resolved today."}
]
{'id': 'c1', 'kind': 'damaged_item_refund', 'detail': 'ORD-123 arrived damaged; customer requests a refund.'}
{'id': 'c2', 'kind': 'duplicate_charge', 'detail': "ORD-456 was charged to the customer's card twice."}
{'id': 'c3', 'kind': 'return_eligibility_question', 'detail': 'Shoes bought on ORD-789 forty days ago; customer asks whether they can still be returned.'}
{'id': 'c4', 'kind': 'chargeback_threat', 'detail': 'Customer will file a chargeback if the issues are not resolv

In [32]:
async def run_all(concerns: list[dict]) -> list[dict]:
    """One coordinator run per concern. Findings come back on the contract."""
    findings = []
    for c in concerns:
        raw = await run(
            f"Concern {c['id']}: {c['detail']}. Customer alice@example.com. "
            f"Today is {TODAY}. Handle only this concern with the right "
            "subagent and return that subagent's JSON findings unchanged.",
            agents=AGENTS, tools=COORDINATOR_TOOLS,
            system_prompt=GOAL_SYSTEM,
        )
        start, end = raw.find("["), raw.rfind("]") + 1
        try:
            findings += json.loads(raw[start:end])
        except (ValueError, json.JSONDecodeError):
            pass  # no finding came back for this concern, so it stays a gap
    return findings


def coverage_gaps(concerns, findings):
    covered = {f.get("concern_id") for f in findings}
    return [c for c in concerns if c["id"] not in covered]


In [33]:
async def resolve(ticket: str, max_rounds: int = 3):
    concerns = await extract_concerns(ticket)
    findings: list[dict] = []
    for round_no in range(1, max_rounds + 1):
        gaps = coverage_gaps(concerns, findings)
        print(f"\nround {round_no}: gaps = {[g['id'] for g in gaps]}")
        if not gaps:
            break
        findings += await run_all(gaps)
    return concerns, findings, coverage_gaps(concerns, findings)


REFUND_LOG.clear()
concerns, findings, remaining = await resolve(ALICE_TICKET)

print("\nconcerns:", [c["id"] for c in concerns])
print("covered: ", sorted({f["concern_id"] for f in findings}))
print("gaps:    ", [g["id"] for g in remaining])


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors



RESULT
 [{"id": "c1", "kind": "damaged_item_refund", "detail": "ORD-123 arrived damaged; customer wants a refund"}, {"id": "c2", "kind": "duplicate_charge", "detail": "ORD-456 was charged to the customer's card twice"}, {"id": "c3", "kind": "return_eligibility", "detail": "Shoes bought on ORD-789 forty days ago; asks whether they can still be returned"}, {"id": "c4", "kind": "chargeback_threat", "detail": "Customer will file a chargeback if the issues are not resolved today"}]

round 1: gaps = ['c1', 'c2', 'c3', 'c4']


⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}


spawned: order-investigator | Look up order ORD-123 and report its state.

Context: Custom
      (inside a subagent)


...


RESULT
 **Concerns identified in this message**

- **c3** — Shoes on ORD-789, purchased ~40 days ago; is a return still possible? → routed to **policy-analyst** (return-window eligibility is a written-policy question). No other concerns present; nothing left unassigned.

**Customer verification:** alice@example.com → CUST-001, Alice Chen, Gold tier. Verified. No order-mutating operation was performed — this was a read-only policy question.

**policy-analyst JSON findings (unchanged):**

```json
{
  "concern": "c3",
  "question": "Is a customer who purchased shoes on order ORD-789 forty (40) days ago (today 2026-09-04) still eligible to return the item?",
  "policy_sections_read": [
    {"section": "2.1", "text": "Standard return window: 30 calendar days from delivery date."},
    {"section": "2.2", "text": "Footwear and apparel may be returned within 45 days if unworn with tags attached."},
    {"section": "2.3", "text": "Electronics return window: 14 days."},
    {"section": "2.4", "

⚠ claude.ai connectors are disabled because ANTHROPIC_API_KEY or another auth source is set and takes precedence over your claude.ai login · Unset it to load your organization's connectors


   tool: mcp__northwind__get_customer {'email': 'alice@example.com'}


spawned: order-investigator | Look up order ORD-123 and report its full state: status, del
      (inside a subagent)


   tool: mcp__northwind__lookup_order {'order_id': 'ORD-123'}
      (inside a subagent)
      (inside a subagent)


The coordinator checks its own output against the ticket, sends the missing concerns back out, and repeats until coverage holds. This only works because Demo 3 gave every finding a `concern_id` to check against.


## Six claims

```
01  The SDK owns the loop so you can own the system
02  A subagent starts empty, so every fact goes in its prompt
03  One job, one tool list, one output contract per subagent
04  The rule that spans agents lives in a hook, not in four prompts
05  A failure comes back with what was attempted and what survived
06  A clean run can still be a wrong answer. Check the decomposition.
```


## Assignment 2

Build a coordinator that resolves a four-concern support ticket.

1. Four specialist subagents, each with an explicit `tools` list.
2. Every subagent gets its facts in its prompt, never by inheritance.
3. Findings returned against the claim, evidence, source contract.
4. One `PreToolUse` hook enforcing the refund ceiling for every agent.
5. One subagent failure returned with partial results and alternatives.
6. A coverage check that re-delegates any concern left unassigned.

Submit the script and one run transcript showing the coverage check firing.
